# Funciones básicas para armar ST-DIP

Author: Evelyn Cueva

Created: 2023-09-02


## 1. MapNet con Flax NNX

Mapnet is a fully connected neural network with relu activations. It takes as input coordinate (2D or 3D) and outputs a feature vector of a specified dimension.

In [5]:
import jax
import jax.numpy as jnp
from flax import nnx
from misc_nnx import MapNetNNX

### Ejemplo de uso de MapNetNNX

El siguiente ejemplo recibe un valor $x$ y entrena MapNetNNX para aproximar la función $y = sin(x).$ 

In [6]:
import optax

# === Datos sintéticos: y = sin(x) con algo de ruido ===
key = jax.random.key(0)
x_train = jax.random.uniform(key, (1024, 1), minval=-jnp.pi, maxval=jnp.pi)
y_train = jnp.sin(x_train) + 0.05 * jax.random.normal(jax.random.split(key)[0], x_train.shape)

# === Modelo y optimizador ===
rngs = nnx.Rngs(42)
model = MapNetNNX(in_features=1, hidden_sizes=(64, 64), out_features=1, rngs=rngs)
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

# === Función de pérdida ===
def loss_fn(model, xb, yb):
    preds = model(xb) # ocupa la definicion de _call_ del modelo
    return jnp.mean((preds - yb) ** 2)

# === Un paso de entrenamiento ===
def train_step(model, optimizer, xb, yb):
    def _loss(m): # generico, es para la derivacion automática
        return loss_fn(m, xb, yb) #solo depende del modelo, datos de entrenamiento y datos verdaderos
    loss, grads = nnx.value_and_grad(_loss)(model) #calculo de gradientes nada que ver con el optimizador
    optimizer.update(model, grads)  # actualiza los parámetros del modelo in-place con el paso del metodo de descenso (optimizer).
    return loss

# === Loop de entrenamiento simple ===
batch_size = 128
n_steps = 2000

for step in range(1, n_steps + 1):
    # muestreo mini-batch
    k1, key = jax.random.split(key)
    idx = jax.random.choice(k1, x_train.shape[0], (batch_size,), replace=False)
    xb, yb = x_train[idx], y_train[idx]

    l = train_step(model, optimizer, xb, yb)

    if step % 200 == 0:
        print(f"step {step:4d} | loss {float(l):.6f}")

# === Prueba rápida ===
x_test = jnp.linspace(-jnp.pi, jnp.pi, 200).reshape(-1, 1)
y_pred = model(x_test)
print("Pred sample:", y_pred[:5].flatten())

2025-10-03 12:00:18.302882: W external/xla/xla/stream_executor/cuda/cuda_command_buffer.cc:780] Retry CUDA graph instantiation after OOM error
E1003 12:00:18.302987 1131298 pjrt_stream_executor_client.cc:3008] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Underlying backend ran out of memory trying to instantiate command buffer with 6 (total of 0 alive graphs in the process). You can try to (a) Give more memory to the driver by reducing XLA_CLIENT_MEM_FRACTION (b) Disable command buffers with 'XLA_FLAGS=--xla_gpu_enable_command_buffer=' (empty set). Original error: Failed to instantiate CUDA graph: CUDA_ERROR_OUT_OF_MEMORY: out of memory


XlaRuntimeError: RESOURCE_EXHAUSTED: Underlying backend ran out of memory trying to instantiate command buffer with 6 (total of 0 alive graphs in the process). You can try to (a) Give more memory to the driver by reducing XLA_CLIENT_MEM_FRACTION (b) Disable command buffers with 'XLA_FLAGS=--xla_gpu_enable_command_buffer=' (empty set). Original error: Failed to instantiate CUDA graph: CUDA_ERROR_OUT_OF_MEMORY: out of memory

In [ ]:
import matplotlib.pyplot as plt

plt.plot(x_test, y_pred, label='Predicción')
plt.scatter(x_test, jnp.sin(x_test), color='r', s=5, label='Verdadero')

## 2. Decoder Net

In [ ]:
import jax, jax.numpy as jnp
from flax import nnx
import optax

### Ejemplo con DecoderNetNNX
Ingresa una matriz de ruido gaussiano de tamaño 8x8 y obtiene una imagen de tamaño 128x128x2.

In [ ]:
from misc_nnx import DecoderNNX

# --------- objetivo sintético suave 128x128x2 ---------
def make_target(H=128, W=128):
    ys = jnp.linspace(-1, 1, H)
    xs = jnp.linspace(-1, 1, W)
    Y, X = jnp.meshgrid(ys, xs, indexing="ij")
    R = jnp.sqrt(X**2 + Y**2)

    # Canal 0: disco gaussiano
    sigma = 0.35
    ch0 = jnp.exp(- (R**2) / (2 * sigma**2))

    # Canal 1: patrón sinusoidal suave
    ch1 = 0.5 * (jnp.sin(3*jnp.pi*X) * jnp.sin(3*jnp.pi*Y)) + 0.5

    target = jnp.stack([ch0, ch1], axis=-1)  # (H, W, 2)
    target = target[None, ...]               # (1, H, W, 2) para NHWC con batch=1
    return target

# --------- setup experimento DIP-like ---------
key = jax.random.key(0)

levels = 3
up = 2
scale_total = up ** (1 + levels)  # 2^(1+3) = 16
H_out = 128
W_out = 128
H_in = H_out // scale_total       # 8
W_in = W_out // scale_total       # 8

assert H_out % scale_total == 0 and W_out % scale_total == 0, "Ajusta niveles o upsampling_factor."

# Entrada (ruido fijo) y objetivo
z = jax.random.normal(key, (1, H_in, W_in, 1))  # NHWC
y_true = make_target(H_out, W_out)              # (1, 128, 128, 2)

# Modelo y optimizador (NNX >= 0.11)
rngs = nnx.Rngs(42)
model = DecoderNNX(in_channels=1, features=64, levels=levels,
                   out_features=2, upsampling_factor=up, rngs=rngs)
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

# Pérdida MSE simple
def loss_fn(m: DecoderNNX, z: jax.Array, y: jax.Array):
    yhat = m(z, training=True)
    return jnp.mean((yhat - y) ** 2)

# Un paso de entrenamiento (API nueva: update(model, grads))
def train_step(m: DecoderNNX, opt: nnx.Optimizer, z: jax.Array, y: jax.Array):
    def _loss(m_):
        return loss_fn(m_, z, y)
    loss, grads = nnx.value_and_grad(_loss)(m)
    opt.update(m, grads)
    return loss

# Loop simple
# Sin batch, todo el objetivo a la vez
n_steps = 1500
for step in range(1, n_steps + 1):
    l = train_step(model, optimizer, z, y_true)
    if step % 150 == 0:
        print(f"step {step:4d} | loss {float(l):.6f}")

# Evaluación (usar promedios corridos en BN)
y_pred = model(z, training=False)
print("Output shape:", y_pred.shape, "| min/max:", float(y_pred.min()), float(y_pred.max()))

In [ ]:
plt.figure(figsize=(8,4))
plt.subplot(1,2,1)
plt.imshow(jnp.abs(y_true[0,:,:,0]), cmap='gray')
plt.title("Canal 0 Verdadero")
plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(jnp.abs(y_pred[0,:,:,0]), cmap='gray')
plt.title("Canal 0 Predicho")
plt.axis('off')

## 3. TD-DIP: MapNet + Decoder

In [1]:
from typing import Sequence, Tuple, Optional
import jax
import jax.numpy as jnp
from flax import nnx
import optax

### 3.1. Ejemplo con MapNetNNX + DecoderNNX

Este ejemplo recibe un vector de tiempos $t$ de tamaño $N$ y pasa por MapNetNNx para obtener una espacio latente de $8\times 8$ y luego pasa por DecoderNNX para obtener una imagen de tamaño $N\times128\times 128 \times 2$.


In [2]:
from misc_nnx import TDIPNNX

# ====== dataset dinámico sintético ======
def make_dyn_target_batch(t_vals: jax.Array, H: int, W: int) -> jax.Array:
    """
    t_vals: (N, 1) en [0,1]
    return: (N, H, W, 2)
    """
    ys = jnp.linspace(-1, 1, H)
    xs = jnp.linspace(-1, 1, W)
    Y, X = jnp.meshgrid(ys, xs, indexing="ij")

    def frame_at_t(t):
        theta = 2 * jnp.pi * t
        cx, cy = 0.3 * jnp.cos(theta), 0.3 * jnp.sin(theta)
        sigma = 0.25
        ch0 = jnp.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sigma**2))
        # patrón sinusoidal que rota con t
        kx, ky = 3.0, 3.0
        ch1 = 0.5 * (jnp.sin(kx*(X*jnp.cos(theta) - Y*jnp.sin(theta)) + 2*jnp.pi*t) *
                     jnp.sin(ky*(X*jnp.sin(theta) + Y*jnp.cos(theta)) + 2*jnp.pi*t)) + 0.5
        return jnp.stack([ch0, ch1], axis=-1)  # (H, W, 2)

    # vmap sobre la primera dimensión de t_vals
    return jax.vmap(lambda tau: frame_at_t(tau[0]))(t_vals)  # (N, H, W, 2)

# ====== configuración del experimento ======
key = jax.random.key(0)

# Dimensiones
levels = 3
up = 2
scale_total = up ** (1 + levels)  # 2^(1+3)=16
px, py = 8, 8                      # tamaño del "latente imagen" que produce MapNet
H_out, W_out = px * scale_total, py * scale_total  # 128x128
assert H_out == 128 and W_out == 128

# Secuencia temporal
N = 31
t_vals = jnp.linspace(0.0, 1.0, N).reshape(N, 1)   # (N, map_in_features=1)
y_true = make_dyn_target_batch(t_vals, H_out, W_out)  # (N, 128, 128, 2)

# ====== modelo + optimizador (NNX >= 0.11) ======
rngs = nnx.Rngs(42)
model = TDIPNNX(
    map_in_features=1,
    map_hidden_sizes=(64, 64),
    cnn_latent_shape=(px, py),
    decoder_features=64,
    momentum=0.9,
    levels=levels,
    out_features=2,              # 2 canales (p.ej., real/imag)
    upsampling_method="nearest",
    upsampling_factor=up,
    rngs=rngs,
)

optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

# ====== pérdida y paso de entrenamiento ======
def loss_fn(m: TDIPNNX, t_batch: jax.Array, y_batch: jax.Array):
    yhat = m(t_batch, training=True)
    return jnp.mean((yhat - y_batch) ** 2)

def train_step(m: TDIPNNX, opt: nnx.Optimizer, t_batch: jax.Array, y_batch: jax.Array):
    def _loss(m_):
        return loss_fn(m_, t_batch, y_batch)
    loss, grads = nnx.value_and_grad(_loss)(m)
    opt.update(m, grads)  # API nueva: (model, grads)
    return loss

# ====== loop (full-batch para simplificar) ======
n_steps = 1500
for step in range(1, n_steps + 1):
    l = train_step(model, optimizer, t_vals, y_true)
    if step % 150 == 0:
        print(f"step {step:4d} | loss {float(l):.6f}")

# ====== evaluación (BN en modo inferencia) ======
y_pred = model(t_vals, training=False)
print("y_pred shape:", y_pred.shape, " | min/max:", float(y_pred.min()), float(y_pred.max()))


2025-10-03 11:50:57.606189: W external/xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 592.56MiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-10-03 11:50:57.979278: W external/xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.01GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-10-03 11:51:08.819058: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 124.00MiB (rounded to 130023424)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation 

ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 130023424 bytes.

#### Visuallización de resultados temporales sintéticos

Dado que la salida del ejemplo tiene dos canales, se visualiza el módulo de la imagen.

In [ ]:
# ===== Video helpers para (N, H, W, C) =====
from typing import Literal, Optional
import numpy as np
import imageio.v3 as iio

def prepare_frames_uint8(
    frames: np.ndarray,
    visualization: Literal["magnitude", "channel"] = "magnitude",
    channel: int = 0,
    normalize: Literal["global", "per_frame"] = "global",
) -> np.ndarray:
    """
    Convierte frames (N,H,W,C) -> (N,H,W) uint8 para video.
    - visualization="magnitude": usa sqrt(ch0^2 + ch1^2) (requiere C>=2).
    - visualization="channel": usa un canal específico (channel=0/1/...).
    - normalize="global": mismo contraste para todo el video.
      "per_frame": normaliza cada frame (útil si varía mucho la intensidad).
    """
    arr = np.asarray(frames)  # por si viene como jax.Array
    if arr.ndim == 3:  # (N,H,W) sin canales
        gray = arr
    else:
        assert arr.ndim == 4, f"Se esperaba (N,H,W,C), obtuve {arr.shape}"
        if visualization == "magnitude":
            assert arr.shape[-1] >= 2, "Se requieren >=2 canales para magnitud."
            gray = np.sqrt(arr[..., 0]**2 + arr[..., 1]**2)
        else:  # "channel"
            gray = arr[..., channel]

    if normalize == "global":
        vmin, vmax = float(gray.min()), float(gray.max())
        if vmax <= vmin:
            gray_u8 = np.zeros_like(gray, dtype=np.uint8)
        else:
            gray_u8 = ((gray - vmin) / (vmax - vmin) * 255.0).clip(0, 255).astype(np.uint8)
    else:  # "per_frame"
        N = gray.shape[0]
        gray_u8 = np.empty_like(gray, dtype=np.uint8)
        for i in range(N):
            g = gray[i]
            vmin, vmax = float(g.min()), float(g.max())
            if vmax <= vmin:
                gray_u8[i] = np.zeros_like(g, dtype=np.uint8)
            else:
                gray_u8[i] = ((g - vmin) / (vmax - vmin) * 255.0).clip(0, 255).astype(np.uint8)

    return gray_u8  # (N,H,W), uint8


def save_gif(frames_u8: np.ndarray, path: str = "video.gif", fps: int = 15):
    """
    Guarda GIF animado con imageio (no requiere ffmpeg).
    """
    assert frames_u8.ndim == 3, "Se espera (N,H,W) uint8"
    duration = 1.0 / fps  # seg por frame
    iio.imwrite(path, frames_u8, extension=".gif", duration=duration, loop=0)
    print(f"GIF guardado en: {path}")


In [ ]:
# y_pred: (N, H, W, 2)  # p. ej., salida de tu TDIPNNX (2 canales = re/imag)
frames_u8 = prepare_frames_uint8(y_pred, visualization="magnitude", normalize="global")

# GIF (no requiere ffmpeg)
save_gif(frames_u8, "tdip.gif")

In [ ]:
frames_u8_true = prepare_frames_uint8(y_true, visualization="magnitude", normalize="global")
save_gif(frames_u8_true, "tdip_true.gif")

### 3.2. Ejemplo con datos reales MRI en numpy

El espacio latente tendrá cierto comportamiento, por ejemplos, puntos de un círculo/hélice/etc. La red es MapNetNNX + DecoderNNX.
Un modelo de T-DIP considera un espacio $k$ sampleado, en este caso, el sampleo es radial. Se incluyen mapas de sensibilidad de las bobinas. Esto cambia principalmente la función de pérdida, que en este caso es la pérdida en el espacio $k$.

#### Cargar los datos disponibles desde el resonador


In [1]:
import jax.numpy as jnp 
from inrmri.radial_acquisitions import RadialAcquisitions
from inrmri.new_radon import get_weight_freqs
from misc_nnx import TDIPNNX

folder = '/home/egcuevaj/repositories/SR-ST-DIP-cMRI/data/Josefa-CineLR/'
print("Loading data...")

data = jnp.load(folder + 'sl5-data.npy')
trajs = jnp.load(folder + 'sl5-traj.npy')
radial_acquisition = RadialAcquisitions(trajs, data)

csmap = jnp.load(folder + 'sl5-csmap.npy')

# Reconstrucción con iterative sense (la reco más básica cuando se conocen las bobinas)
reco_sense = jnp.load(folder + 'sl5-reco-sense.npy')

# Reconstrucción con GRASP (una reco que además de las bobinas incluye regularización (TV en tiempo))
reco_grasp = jnp.load(folder + 'sl5-reco-grasp.npy')

print("Data loaded!")
print(radial_acquisition)
print(f"csmap has shape {csmap.shape}: {csmap.shape[0]} sensitivity maps of size {csmap.shape[1:3]}")
print(f"reco_sense has shape: {reco_sense.shape}: {reco_sense.shape[-1]} frames of size {reco_sense.shape[:2]}")
print(f"reco_grasp has shape: {reco_grasp.shape}: {reco_grasp.shape[-1]} frames of size {reco_grasp.shape[:2]}")

Loading data...
Data loaded!
Radial adquisition: 28 coils, 31 frames of 15 spokes each, 256 samples per spokes.
csmap has shape (28, 256, 256): 28 sensitivity maps of size (256, 256)
reco_sense has shape: (256, 256, 31): 31 frames of size (256, 256)
reco_grasp has shape: (256, 256, 31): 31 frames of size (256, 256)


Se reorganizan los datos para que sean compatibles con el modelo y la función de pérdida. Esto lo realiza la función `radial_acquisition.generate_dataset()`.

Returns
- dataX: Trayectorias de muestreo con información temporal añadida.

         Forma: `(n_frames * n_spokes_per_frame, n_samples + 1, 2)`.

- dataY : Datos de Fourier adquiridos en k-space.
        
         Forma: `(n_frames * n_spokes_per_frame, n_coils, n_samples, 1)`.

In [2]:
print("Creating dataset...")
train_X_full, train_Y = radial_acquisition.generate_dataset()
train_Y = 100 * train_Y  # Escalado de la señal para mantener un orden de magnitud adecuado
print(train_X_full.shape, train_Y.shape)

Creating dataset...
(465, 257, 2) (465, 28, 256, 1)


In [3]:
from inrmri.radial_acquisitions import kFOV_limit_from_spoke_traj, check_correct_dataset

# -------------------------------------------------------------------
# Chequeo de las trayectorias
# -------------------------------------------------------------------
check_correct_dataset(train_X_full)

# Escalamiento inicial
scaling_param = 1.0

The trajectories are centered


In [4]:
from inrmri.radon import calculate_angle
from jax import vmap
import jax.numpy as jnp

# Los spokes (kx,ky) pueden representarse solo con alpha, se elimina una dimensión usando eso.
angles = vmap(calculate_angle)(train_X_full[:, 1:]) # hay nframes*n_spokes angles
# Tiempos asociados a cada frame (almacenados en la primera columna de cada spoke)
times = train_X_full[:, 0, 1] 
# times = train_X[:, 0, 0] tiene la misma información 
# Nueva representación de entradas: (angle, time)
train_X = jnp.stack([angles, times], axis=-1)
print("New train_X shape:", train_X.shape)  # (nframes*nspokes, 2)

New train_X shape: (465, 2)


In [5]:
# -------------------------------------------------------------------
# Configuración inicial
# -------------------------------------------------------------------
gtim = reco_grasp # es una imagen en cine 31 frames
print(gtim.shape)
NFRAMES = gtim.shape[-1]          # número de frames dinámicos
IMSHAPE = gtim.shape[:2]          # dimensiones espaciales (px, py)
N = IMSHAPE[0]                    # dimensión lateral (asumiendo cuadrado)
total_cycles = 1                  # número total de ciclos cardiacos en la adquisición

(256, 256, 31)


In [6]:
# Grid temporal normalizado (0 a 1, excluyendo el endpoint)
ts = jnp.linspace(0, 1, gtim.shape[-1], endpoint=False)

#### Estructura de la red con NNx

Input: Puntos de una hélice para un ciclo cardíaco (será como un círculo)

In [7]:
# ================== Helix (latente temporal) ==================
def helix_generator(nframes: int, total_cycles: float):
    ts = jnp.linspace(0,total_cycles, nframes, endpoint=False)
    helix = jnp.stack([jnp.cos(2 * jnp.pi * ts), jnp.sin(2 * jnp.pi *ts), ts/total_cycles], axis=-1)
    return helix


helix = helix_generator(NFRAMES, total_cycles) #(31, 3) 
d = helix.shape[1]

def get_latent(latent, t_index):
    """
    Obtiene la representación latente asociada a un frame.

    Parameters
    ----------
    t_index : int or None
        Índice temporal. Si es None, devuelve la latente completa.

    Returns
    -------
    latent : np.ndarray
        Representación latente de forma (N,) o (nframes, N).
    """
    if t_index is None:
        return latent
    return latent[t_index, :]

#### Función de pérdida

Vamos a definir la función de pérdida, dado que la pérdida se calcula en el espacio $k$, se usa la transformada de Radon para llegar al espacio de Fourier y comparar con los datos adquiridos (Fourier slice Theorem).

In [8]:
import jax.numpy as jnp
import jax
from flax import nnx
from jax.scipy.interpolate import RegularGridInterpolator as RGI
from misc_nnx import ForwardRadonOperator
from misc_nnx import weighted_loss

In [9]:
# ================== Loss NNX (TDIPNNX con salida Re/Im) ==================
def loss_fn(model: "TDIPNNX",
            X: jnp.ndarray,          # (B, 2): [alpha_rad, time_norm]
            Y: jnp.ndarray,          # (B, C, N) complejo o (B,C,N,1)/(B,C,N,2)
            helix: jnp.ndarray,      # (NFRAMES, d)
            radon_operator,
            weight_freqs: jnp.ndarray):  # (N,)

    alphas = X[:, 0]
    times  = X[:, 1]
    t_idx = jnp.int32(times * NFRAMES)
    latente = helix[t_idx, :]

    # Forward (BN en training=True)
    pred_images = model(latente, training=True) # (B, H, W, 1) salen transformada en un complejo
    pred_images = pred_images[..., 0]  # quitar la dimensión extra de canales → (batch, H, W)

    # Aplicar operador de Radon con ponderación de sensibilidad de coils
    rotated_im = radon_operator.rotated_csweighted_ims(pred_images, alphas)   # (batch, n_coils, N)
    spoke_radon_kspace = radon_operator.radon_transform(rotated_im)          # (batch, n_coils, N)

    # Ground truth: quitar dimensión extra en Y
    target_kspace = Y[..., 0]  # (batch, n_coils, N)
    w = (1.0 + weight_freqs)[None, None, :] # (1,1,N)

    return weighted_loss(spoke_radon_kspace, target_kspace, w)


In [10]:

# ================== Paso de entrenamiento ==================
def make_train_step(radon_op,
                    helix: jnp.ndarray,
                    weight_freqs: jnp.ndarray):
    @nnx.jit 
    def train_step(model: "TDIPNNX", optimizer: nnx.Optimizer, Xb: jnp.ndarray, Yb: jnp.ndarray):
        def _loss(m):
            return loss_fn(m, Xb, Yb, helix, radon_op, weight_freqs)
        loss, grads = nnx.value_and_grad(_loss)(model)
        optimizer.update(model, grads) 
        return loss
    return train_step

# ================== Mini-batching ==================
def sample_minibatch(X: jnp.ndarray, Y: jnp.ndarray, batch_size: int, key: jax.Array):
    n = X.shape[0]
    idx = jax.random.choice(key, n, (batch_size,), replace=False)
    return X[idx], Y[idx]

In [11]:
print(train_X.shape, train_Y.shape)  # (nframes*nspokes, 2), (nframes, C, N) o (nframes, C, N, 1/2)
print(csmap.shape)  # (C, N, N)

(465, 2) (465, 28, 256, 1)
(28, 256, 256)


#### Entrenamiento

In [12]:
import os
os.environ["XLA_FLAGS"] = os.environ.get("XLA_FLAGS","") + " --xla_gpu_strict_conv_algorithm_picker=false"
# Optional: reduce prealloc to leave headroom for workspaces
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Now import jax, flax, etc.

In [13]:
import optax
from misc_nnx import ForwardRadonOperator, TDIPNNX

# 1) Construye operador y latente
NFRAMES = 31
total_cycles = 1.0
helix = helix_generator(NFRAMES, total_cycles)

# Cálculo del límite de FOV a partir de un spoke de ejemplo
spoke_limit = kFOV_limit_from_spoke_traj(train_X_full[0, 1:, :])
radon_operator = ForwardRadonOperator(csmap, spoke_limit)

# 2) Modelo
rngs = nnx.Rngs(0)
model = TDIPNNX(
    map_in_features=helix.shape[1],   # d = 3 de la hélice
    map_hidden_sizes=(64, 64),
    cnn_latent_shape=(8, 8),          # ajusta según tu decoder: H_out = 8 * 2**(1+levels)
    decoder_features=64,
    momentum=0.9,
    levels=4,
    out_features=2,                   # Re/Im
    upsampling_method="nearest",
    upsampling_factor=2,
    rngs=rngs,
)

# 3) Optimizador 
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

# 4) Entrenamiento
WEIGHT_FREQS = get_weight_freqs(N)
train_step = make_train_step(radon_operator, helix, jnp.asarray(WEIGHT_FREQS))
 
key = jax.random.key(0)
batch_size = 3
n_steps = 1_000

In [14]:
import jax, jax.numpy as jnp
import flax.nnx as nnx
from tqdm import tqdm

def snapshot_best_params(model):
    # Toma solo los parámetros (nnx.Param) y los convierte a arrays puros
    return nnx.pure(nnx.state(model, nnx.Param))  # -> nnx.State de jax.Array

def restore_params_(model, params_state):
    # Escribe los arrays al modelo (in-place)
    nnx.update(model, params_state)

best_loss   = float("inf")
best_params = None
best_step   = -1

for step in tqdm(range(n_steps+1), desc="train iter", leave=True):
    key, ks = jax.random.split(key)
    Xb, Yb  = sample_minibatch(jnp.asarray(train_X), jnp.asarray(train_Y), batch_size, ks)

    train_loss_value = train_step(model, optimizer, Xb, Yb)  # actualiza model in-place

    loss_float = float(train_loss_value)
    if loss_float < best_loss:
        best_loss   = loss_float
        best_step   = step
        best_params = snapshot_best_params(model)

    if step % 100 == 0:
        print(f"step {step:4d} | loss {loss_float:.6f} | best@{best_step}={best_loss:.6f}")

# Antes de validar / exportar: restaura el mejor estado
if best_params is not None:
    restore_params_(model, best_params)


train iter:   0%|          | 0/1001 [00:00<?, ?it/s]

256


2025-10-05 09:26:43.380044: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-05 09:26:43.639998: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-05 09:26:43.847652: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-05 09:26:43.972495: E external/xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
train iter:   0%|          | 3/1001 [00:12<55:25,  3.33s/it]  

step    0 | loss 0.748119 | best@0=0.748119


train iter:  10%|█         | 103/1001 [00:17<00:41, 21.62it/s]

...

step  800 | loss 0.408509 | best@378=0.314851


train iter:  90%|█████████ | 904/1001 [00:54<00:04, 21.18it/s]

step  900 | loss 0.357815 | best@378=0.314851


train iter: 100%|██████████| 1001/1001 [00:59<00:00, 16.75it/s]

step 1000 | loss 0.362266 | best@378=0.314851


#### Visualización de resultados MRI reales

In [15]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx

def cine_from_params(
    model: "TDIPNNX",
    best_params: nnx.State,          # devuelto por snapshot_best_params(model)
    helix: jnp.ndarray,              # (NFRAMES, d)
    t_idx,                           # (Nt,)
    batch: int
) -> jnp.ndarray:
    """
    Evalúa la cine usando un 'snapshot' de los mejores parámetros (best_params)
    sin modificar permanentemente el estado actual del modelo.

    Pasos:
      1) Guarda los parámetros actuales (nnx.Param) del modelo.
      2) Restaura 'best_params' al modelo (in-place).
      3) Ejecuta inferencia por lotes con BN en modo inferencia (training=False).
      4) Restaura los parámetros originales.

    Retorna:
      predim_cplx: jnp.ndarray de forma (H, W, Nt) en complejo.
    """
    # 1) Guardar params actuales
    current_params = nnx.pure(nnx.state(model, nnx.Param))

    try:
        # 2) Cargar mejores params
        nnx.update(model, best_params)

        # 3) Inferencia por lotes (igual a tu cine_from_trained)
        t_idx = jnp.asarray(t_idx)
        Nt = t_idx.shape[0]

        # Padding para completar batch
        pad = (-Nt) % batch
        if pad:
            t_idx = jnp.concatenate([t_idx, jnp.full((pad,), t_idx[-1])], axis=0)

        # (num_batches, batch)
        stacked_idx = t_idx.reshape(-1, batch)

        def eval_batch(idx_row):
            t_lat = helix[idx_row]                  # (batch, d)
            y = model(t_lat, training=False)        # (batch, H, W, 2) Re/Im
            return y

        y_batched = jax.lax.map(eval_batch, stacked_idx)          # (B, batch, H, W, 2)
        y_full = y_batched.reshape(-1, *y_batched.shape[2:])      # (Nt+pad, H, W, 2)
        y_full = y_full[:Nt]

        y_cplx = y_full#[..., 0] + 1j * y_full[..., 1]             # (Nt, H, W)
        predim_cplx = jnp.moveaxis(y_cplx, 0, -1)                 # (H, W, Nt)
        return predim_cplx

    finally:
        # 4) Restaurar params originales del modelo
        nnx.update(model, current_params)


In [16]:
if best_params is not None:
    predim = cine_from_params(model, best_params, helix, jnp.arange(NFRAMES), batch=5)

In [19]:
from inrmri.utils import is_inside_of_radial_lim, meshgrid_from_subdiv_autolims

# Operador de Radon y grilla espacial
grid = meshgrid_from_subdiv_autolims(IMSHAPE)

# -------------------------------------------------------------------
# Post-procesamiento de reconstrucciones
# -------------------------------------------------------------------
def post_processing(im):
    """
    Aplica una máscara espacial para limitar la reconstrucción
    al dominio válido definido por la adquisición radial.

    Parameters
    ----------
    im : np.ndarray
        Imagen reconstruida.
        Forma: (px, py, nframes)

    Returns
    -------
    np.ndarray
        Imagen postprocesada, con voxeles fuera del FOV puestos en cero.
    """
    # Calcula la máscara booleana de puntos dentro del límite radial
    inside_mask = is_inside_of_radial_lim(grid, 1.0) #radio 1

    # Expande la máscara a lo largo de la dimensión temporal
    return im * inside_mask[:, :, None]
# Se aplica la máscara espacial y se normaliza a [0,1] por magnitud
print("Reconstrucción antes de post-processing: ", predim.shape)
predim = post_processing(predim) / jnp.abs(predim).max() 

print("Reconstrucción final: ", predim.shape)

Reconstrucción antes de post-processing:  (256, 256, 1, 31)


2025-10-05 09:32:57.649707: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.88GiB (rounded to 4160749568)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-10-05 09:32:57.650546: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] **********************************************************************______________________________
E1005 09:32:57.650579 3452851 pjrt_stream_executor_client.cc:3008] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 4160749568 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 4160749568 bytes.

In [ ]:
from inrmri.basic_plotting import full_halph_FOV_space_time
from inrmri.image_processor import BeforeLinRegNormalizer, reduce_crop_abs

crop_ns = [35,30,20,38]
gtim = jnp.load(folder + 'sl5-reco-reference.npy') # 
improc = BeforeLinRegNormalizer(gtim, [0,0,0,0]) # normaliza todas las imagenes a la escala de gtim 

gt_proc = improc.process(gtim)

def final_processing(im):
    im = improc.process(im)
    im = jnp.roll(im, 15, axis=-1) # 15 es el argmin del error con gastao luego de probar todos los rolls 
    return im 
 
# el [1:] es porque gtim solo tiene 30 frames (no 31)
pred_proc = final_processing(predim[...,1:]) 
sense_proc = final_processing(reco_sense[...,1:])
grasp_proc = final_processing(reco_grasp[...,1:])
total_kiter = n_steps
frame = 8 # 8, 23 
vmax = jnp.abs(gtim).max()
recos = [gt_proc, pred_proc, grasp_proc, sense_proc]
titles = ['MCCINE (Gastao)', f'TD-DIP {total_kiter}k', 'GRASP', 'SENSE']
fig, axs = full_halph_FOV_space_time(recos, crop_ns, frame = frame,saturation=0.5, vmax=vmax)
for ax, title in zip(axs, titles):
    ax[0].set_title(title)

## 4. Multi-slice + TD-DIP 

In [34]:
import numpy as onp
import jax.numpy as jnp
from inrmri.utils_rdls import safe_normalize, get_center,  pad_axis_to_length
from inrmri.utils_rdls import create_folder, save_frames_as_gif_with_pillow
from misc_nnx import sample_from_groups
from inrmri.utils_rdls import get_varying_keys
import itertools

#### Cargar datos

In [35]:
slices_list = [1, 2, 3, 4, 5, 6, 7, 8]
total_slices = len(slices_list)

n_coils = 15
num_frames = 30
saturation = 0.3
val_frames = jnp.array([0], dtype=jnp.int32)

volunteer = 'MP'
dataset = 'DATA_0.55T'
base_path = '/mnt/workspace/datasets/pulseqCINE/'

base_folder = base_path + dataset + '/' + volunteer + '/'
train_data_folder = base_folder + 'traindata/'
model_path = 'stDIP/'


In [36]:
experiment_params = {              
    'N':                    [256],                 # elements of the readout, lenght of the spoke
    'mapnet_layers':        [16],                  # MapNet: layers
    'cnn_latent_shape':     [8],                   # Generatior: input shape, is the trained latent representation by MapNet
    'levels':               [4],                   # Generatior: levels of up sampling
    'features':             [16],                  # Generatior: features
    'iter':                 [500],                # number of iterations
    'bs':                   [1],                   # Number of cardiac phases per iteration (real batch size: bs * nspokes * nslices)
    'addConst':             [False],               # if add a constant to the fixed manifold
    'str_filter':           ['ramp'],              # frequency weighting
    'denoise_type':         ['tv'],                # Regularization ('tv', 'l1', 'tikhonov')
    'lambda':               [0],                   # lambda for regularization
    'lr_schedule':          ['constant_schedule'], # learning rate scheduler
    'lr_init_value':        [5e-3],                # learninig rate start (or constant value depending on the scheduler)
    'lr_end':               [1e-3],                # learning rate end (optional depending on the scheduler)
    'lr_transition_steps':  [3000],           # how many steps to decay over
    'lr_decay_rate':        [0.90],           # (parameter for 'exponential_decay' scheduler)
    'lr_power':             [20],             # (parameter for 'polynomial_schedule' scheduler)
    'metric_step':          [20],             # step of iterations to compute metrics
    'window_size':          [5],              # number of elements in windows to compute window metrics (variance)
    'nspokes':              [1],              # number of spokes used of each frame in the nspokes-wise training
    'select_by':           ['loss', 'mean_var'],  # The used metric for selecting the best parameters
    'nr_iqm':               [True],           # No-reference image quality metric (WMV)
    'fr_iqm':               [True],           # Full-reference image quality metric (PSNR, SSIM, AP)
    'debug':                [False],          # If debug, multiple shapes and time should be printed
    'data_ponderator':      [1],              # currently the data is normalized, so this ponderator would ponderate a normalized data

}


In [37]:
# --- create the multislice dataset ---
Y_data_list, X_data_list, csm_list = [], [], []
spclim_list      =   []
hollow_mask_list =   []
recon_fs_list    =   []
exp_folder_path_list  =   []

for  slice_num in slices_list:
    dataset_name = 'slice_' + str(slice_num) + '_' + str(total_slices) +'_nbins' + str(num_frames)
    path_file = train_data_folder + dataset_name + '.npz'
    data = onp.load(path_file)
    y_data_item = data['Y_data']
    x_data_item = data['X_data']
    csm_item    = data['csm']
    spclim_item = data['spclim']
    y_data_item = data['Y_data']
    if dataset == 'DATA_0.55T':
        hollow_mask_item  = data['hollow_mask']    
    else:
        hollow_mask_item  = data['hollow_mask_computed']   

    # Check coils
    y_data_item = pad_axis_to_length(y_data_item, 1, n_coils, pad_value=0+0j)
    csm_item = pad_axis_to_length(csm_item, 0, n_coils, pad_value=0+0j)

    # append items in lists
    Y_data_list.append(y_data_item)
    X_data_list.append(x_data_item)
    csm_list.append(csm_item)
    spclim_list.append(spclim_item)
    hollow_mask_list.append(hollow_mask_item)

    recon_fs = data['recon_fs']
    recon_fs = get_center(recon_fs)
    recon_fs = safe_normalize(recon_fs)
    recon_fs_list.append(recon_fs[:, :, val_frames])
    save_folder = 'results/' + model_path + dataset_name.replace(".", "_") + '/'
    create_folder(save_folder, reset=False)
    save_frames_as_gif_with_pillow(save_folder, recon_fs, filename='recon_fs', vmax=1, saturation=saturation, fps=30)

    exp_folder_path = save_folder + 'test'  + '/'
    exp_folder_path_list.append(exp_folder_path)
    create_folder(exp_folder_path, reset=False)

hollow_mask_array = jnp.stack(hollow_mask_list, axis=0)

In [38]:
# Chequear los datos
print("Y_data_list length:", len(Y_data_list))
print("X_data_list length:", len(X_data_list))
print("csm_list length:", len(csm_list))
print("hollow_mask_list length:", len(hollow_mask_list))
print("X_data_list[0] shape:", X_data_list[0].shape)
print("Y_data_list[0] shape:", Y_data_list[0].shape)
print("csm_list[0] shape:", csm_list[0].shape)
print("hollow_mask_list[0] shape:", hollow_mask_list[0].shape)

Y_data_list length: 8
X_data_list length: 8
csm_list length: 8
hollow_mask_list length: 8
X_data_list[0] shape: (243, 2)
Y_data_list[0] shape: (243, 15, 256, 1)
csm_list[0] shape: (15, 256, 256)
hollow_mask_list[0] shape: (256, 256)


In [39]:
import matplotlib.pyplot as plt

X_test = X_data_list[0]
print(X_test)

[[1.02825928 0.        ]
 [1.44064844 0.        ]
 [1.85303712 0.        ]
 [2.2654264  0.        ]
 [2.67781591 0.03333333]
 [3.09020424 0.03333333]
 [3.50259304 0.03333333]
 [3.91498184 0.03333333]
 [4.32737255 0.06666667]
 [4.73975992 0.06666667]
 [5.15214968 0.06666667]
 [5.56453705 0.06666667]
 [5.97692728 0.06666667]
 [0.10613009 0.1       ]
 [0.51851916 0.1       ]
 [0.9309091  0.1       ]
 [1.34329629 0.1       ]
 [5.10618448 0.13333333]
 [5.51857471 0.13333333]
 [1.75568557 0.13333333]
 [2.16807461 0.13333333]
 [2.58046317 0.13333333]
 [2.99285245 0.13333333]
 [3.40524244 0.13333333]
 [5.93096399 0.16666667]
 [0.06016656 0.16666667]
 [0.47255522 0.16666667]
 [0.88494349 0.16666667]
 [1.29733312 0.16666667]
 [3.81763005 0.16666667]
 [4.23001909 0.16666667]
 [4.64240789 0.16666667]
 [5.05479765 0.16666667]
 [1.7097224  0.2       ]
 [2.12211061 0.2       ]
 [2.53449869 0.2       ]
 [2.9468894  0.2       ]
 [5.46718597 0.2       ]
 [5.87957382 0.2       ]
 [0.00877822 0.2       ]


In [40]:
from inrmri.new_radon import get_weight_freqs, make_forward_radon_operator

#### Cargar parametros y operadores

In [41]:
## --- Radon operator list ---
radon_operator_list = []
for jj in range(len(csm_list)):
    radon_operator_list.append(make_forward_radon_operator(csm_list[jj], spclim_list[jj]))

In [42]:
h_params = experiment_params
n_slices = len(slices_list)

In [43]:
from misc_nnx import multi_slice_circle_generator
import optax
import jax
from misc_nnx import TDIPNNX
import flax.nnx as nnx

# add information
h_params['NFRAMES']    = num_frames
h_params['n_slices']   = n_slices
h_params['val_frames'] = val_frames

CONFIG_NET = {
    'mapnet_layers':    [h_params['mapnet_layers'], h_params['mapnet_layers'],],
    'cnn_latent_shape': (h_params['cnn_latent_shape'],h_params['cnn_latent_shape']),
    'levels':            h_params['levels'],
    'features':          h_params['features']
}

# data ponderation
Y_data_list_current     =  []
recon_fs_list_current   =  []
for i in range(len(Y_data_list)):
    Y_data_list_current.append(Y_data_list[i])
    recon_fs_list_current.append(recon_fs_list[i]) #*  h_params['data_ponderator']   )

# 1) Construye operador y latente
total_cycles = 1.0
radius = 1.0
z_min = -10
z_max = 10
key_latent = jax.random.PRNGKey(0)
addConst = False
circles = multi_slice_circle_generator(num_frames, n_slices, key_latent, addConst, radius=radius, z_min=z_min, z_max=z_max)

In [44]:
circles.shape

(8, 30, 3)

#### Modelo

In [45]:
# 2) Modelo
rngs = nnx.Rngs(0)
model = TDIPNNX(
    map_in_features=circles.shape[2],   # d = 3 en circulos 
    map_hidden_sizes=(64, 64),
    cnn_latent_shape=(8, 8),          # ajusta según tu decoder: H_out = 8 * 2**(1+levels)
    decoder_features=64,
    momentum=0.9,
    levels=4,
    out_features=2,                   # Re/Im
    upsampling_method="nearest",
    upsampling_factor=2,
    rngs=rngs,
)

# 3) Optimizador 
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

#### Función de pérdida

In [ ]:
import jax
import jax.numpy as jnp
from jax import vmap
from flax import nnx
from functools import partial
from misc_nnx import weighted_loss, to_complex

weights = (1. + WEIGHT_FREQS)[None, None, :]  # (1,1,nsamples)

def loss_fn(
    model: "TDIPNNX",
    X_list_batch, 
    Y_list_batch, 
    circles: jnp.ndarray,      # (Nslices, NFRAMES, d)
    weight_freqs: jnp.ndarray  # (NX,)
):
    
    n_slices = len(X_list_batch)
    NFRAMES  = circles.shape[1] # (8,30,3)

    def per_spoke_loss(im_1hw, x_spoke, y_spoke, radon_operator):
        # im_1hw: (1, H, W) ; x_spoke: (features,) -> usamos x_spoke[0] = alpha
        # y_spoke: (CMAP, NX, 1)
        print(f"la imagen para el operador de radon es {im_1hw.shape}")
        alphas_frame = x_spoke[0:1]         # (1,)
        y_data = y_spoke[..., 0]            # (1, CMAP, NX)

        pred_kspace = radon_operator(im_1hw, alphas_frame)  # (1, CMAP, NX)

        
        # weighted_loss(pred, target, w) -> escalar
        return weighted_loss(pred_kspace, y_data, weights)

    # per_frame_loss recibe una imagen que pertenece a un batch de una slice
    def per_frame_loss(imsb_f, Xsb_f, Ysb_f, radon_operator_s):
        # imsb_f tiene tamaño (1, Nx, Ny), Xsb_f es (1, 2), Ysb_f es (1, 15,256,1)
        imsb_f = imsb_f[None, ...]
        spoke_loss = vmap(lambda xs, ys: per_spoke_loss(imsb_f, xs, ys, radon_operator_s))
        return jnp.sum(spoke_loss(Xsb_f, Ysb_f))  # escalar

    total_loss = 0.0
    for slice in range(n_slices):
        
        X_sb = X_list_batch[slice] # (batch, 2)       
        Y_sb = Y_list_batch[slice] # (batch, CMAP, samples, 1)
        radon_operator_s = radon_operator_list[slice]
        print("entrando a los datos de una slice")
        print(X_sb.shape)
        print(Y_sb.shape)


        times   = X_sb[..., 1]                     # (NFRAMES, NSP)
        times_batch_indx = jnp.clip(jnp.floor(times * NFRAMES).astype(jnp.int32), 0, NFRAMES-1)
        frame_latents = circles[slice, times_batch_indx, :]   # (NFRAMES, d)

        # Reconstrucciones para un slice y un batch
        recon_ims_sb = model(frame_latents, training=True)      # (batch, H, W)
        recon_ims_sb = to_complex(recon_ims_sb)
        # y = y[..., 0]
        # frame_loss se va a aplicar a un conjunto de imagenes de un batch en un slice específica
        # per_frame_loss calcula para cada frame del batch, cada frame tiene un alpha asociado (radon_operator_s)
        # radon_operator solo depende del slice (por los coil sensitivity maps) pero radon_transform necesitará
        # alpha que va almacenado en Xsb_f
        def frame_loss_fn(imsb_f, Xsb_f, Ysb_f): # frames del batch (de 1 en 1)
            frame_loss = per_frame_loss(imsb_f, Xsb_f, Ysb_f, radon_operator_s) # (256,256), (1,2), (1,15,256,1), rad_op
            return frame_loss
        print("ims shape:", recon_ims_sb.shape, "X shape:", X_sb.shape, "Y shape:", Y_sb.shape)
        total_loss += jnp.sum(vmap(frame_loss_fn)(recon_ims_sb, X_sb, Y_sb))
        total_loss /= n_slices
    print("La funcion de perdida es")
    print(total_loss)
    return total_loss

In [62]:
# ================== Paso de entrenamiento ==================
def make_train_step(circles, weight_freqs):
    @nnx.jit 
    def train_step(model: "TDIPNNX", optimizer: nnx.Optimizer, Xb_list, Yb_list):
        def _loss(m):
            return loss_fn(m, Xb_list, Yb_list, circles, weight_freqs)
        loss, grads = nnx.value_and_grad(_loss)(model)
        optimizer.update(model, grads) 
        return loss
    return train_step

In [ ]:
import jax, jax.numpy as jnp
import flax.nnx as nnx
from tqdm import tqdm 


def snapshot_best_params(model):
    # Toma solo los parámetros (nnx.Param) y los convierte a arrays puros
    return nnx.pure(nnx.state(model, nnx.Param))  # -> nnx.State de jax.Array

def restore_params_(model, params_state):
    # Escribe los arrays al modelo (in-place)
    nnx.update(model, params_state)

# 4) Entrenamiento

best_loss   = float("inf")
best_params = None
best_step   = -1

N = 256
WEIGHT_FREQS = get_weight_freqs(N)
train_step = make_train_step(circles, jnp.asarray(WEIGHT_FREQS))
 
key = jax.random.key(0)
batch_size = 1
n_steps = 1000
epsilon = 1e-4
nspokes = 1

for step in tqdm(range(n_steps), desc='train iter', leave=True):
    key, key_batch, key_loss = jax.random.split(key, 3)

    # sample frames
    idx_batch_latent = jax.random.choice(key_batch, num_frames, shape=(batch_size,), replace=False)
    times_batch      = idx_batch_latent / num_frames
    index_frames     = jnp.int32(times_batch * num_frames)

    # build nspokes-wise, per-slice batches
    Xb, Yb = [], []
    for item in range(len(X_data_list)):
        X = X_data_list[item]
        Y = Y_data_list[item]
        idx_groups = [jnp.where(jnp.abs(X[:, 1] - t) < epsilon)[0] for t in times_batch]
        group_inds = sample_from_groups(idx_groups, nspokes, key)
        Xb.append(jnp.stack([X[g] for g in group_inds]))
        Yb.append(jnp.stack([Y[g] for g in group_inds]))
    X_batch_array = jnp.stack(Xb)
    Y_batch_array = jnp.stack(Yb)
    train_loss_value = train_step(model, optimizer, X_batch_array, Y_batch_array)  # actualiza model in-place

    loss_float = float(train_loss_value)
    if loss_float < best_loss:
        best_loss   = loss_float
        best_step   = step
        best_params = snapshot_best_params(model)

    if step % 100 == 0:
        print(f"step {step:4d} | loss {loss_float:.6f} | best@{best_step}={best_loss:.6f}")

# Antes de validar / exportar: restaura el mejor estado
if best_params is not None:
    restore_params_(model, best_params)


train iter:   0%|          | 0/1000 [00:00<?, ?it/s]

entrando a los datos de una slice
(1, 1, 2)
(1, 1, 15, 256, 1)
ims shape: (1, 256, 256) X shape: (1, 1, 2) Y shape: (1, 1, 15, 256, 1)
la imagen para el operador de radon es (1, 256, 256)
se va a rotar una imagen de tamano (15, 1, 256, 256) en angulos VmapTracer<float32[1]>
entrando a los datos de una slice
(1, 1, 2)
(1, 1, 15, 256, 1)
ims shape: (1, 256, 256) X shape: (1, 1, 2) Y shape: (1, 1, 15, 256, 1)
la imagen para el operador de radon es (1, 256, 256)
se va a rotar una imagen de tamano (15, 1, 256, 256) en angulos VmapTracer<float32[1]>
entrando a los datos de una slice
(1, 1, 2)
(1, 1, 15, 256, 1)
ims shape: (1, 256, 256) X shape: (1, 1, 2) Y shape: (1, 1, 15, 256, 1)
la imagen para el operador de radon es (1, 256, 256)
se va a rotar una imagen de tamano (15, 1, 256, 256) en angulos VmapTracer<float32[1]>
entrando a los datos de una slice
(1, 1, 2)
(1, 1, 15, 256, 1)
ims shape: (1, 256, 256) X shape: (1, 1, 2) Y shape: (1, 1, 15, 256, 1)
la imagen para el operador de radon es (